In [1]:
from os import listdir
from typing import TypedDict

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import PatchCollection
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from PIL import Image
from skimage.transform import resize
from sklearn.preprocessing import StandardScaler

from dinosaw.linear_probe import (
    RampTypes,
    Regressor,
    gen_sample_mask,
    get_model as get_reg_model,
    get_ramp,
)
from dinosaw.utils import add_custom_font, get_features
from dinosaw.wrappers import ModelTypes
from dinosaw.wrappers.register_models import get_model_

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/Documents/academic/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_model: ModelTypes = "tr_dinov2_s"
model = get_model_(selected_model, DEVICE, True, "../../models/checkpoints", conf_dir="../../models/dinov3")
models = {selected_model: model}

n_dims = model.embed_dim

2026-11-08 13:03:52 | I | factory.py                 : 180 | Building wrapper 'tr_dinov2_s' on device cuda:0
2026-11-08 13:03:52 | I | factory.py                 : 158 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-11-08 13:03:53 | I | wrapper.py                 :  57 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12


In [3]:
ds_folder = 'data/linear_probe/homog_micros'
MAX_IMGS: int | None = 60  # Set to 30 for the 30-image subset, or None to use all images in ds_folder
IMG_SIZE: tuple[int, int] = (518, 518)  # Resize so train/test patch masks align across images A and B
replace_with_random_noise: bool = False

image_files = sorted(listdir(ds_folder))
if MAX_IMGS is not None:
    image_files = image_files[:MAX_IMGS]
n_imgs = len(image_files)

# Featurise each image once
features: list[np.ndarray] = []
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (*IMG_SIZE, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')
    feats = get_features(model, img, channel_last=True)
    features.append(np.nan_to_num(feats, nan=0.0))

2026-11-08 13:03:53 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:54 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:54 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,

In [4]:
preview_folder = 'data/linear_probe/'
preview_img_names = ['black_square_518.png', 'bulldog_518.png', 'edinburgh.png', '000.png']
preview_feats: list[np.ndarray] = []
preview_imgs: list[Image.Image] = []

for fname in preview_img_names:
    img_path = f'{preview_folder}/{fname}'
    img = Image.open(img_path).convert('RGB')
    img = img.resize(IMG_SIZE)
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (*IMG_SIZE, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')
    feats = get_features(model, img, channel_last=True)
    preview_imgs.append(img)
    preview_feats.append(feats)

2026-11-08 13:03:59 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:59 | I | wrapper.py                 : 102 | Processing image, size: [518, 518]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-11-08 13:03:59 | I | wrapper.py                 : 213 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,

In [5]:
class CrossImageProbeResult(TypedDict):
    # R^2 scores across the 4 evaluation sets:
    # 1) A_train: train on A, apply to same train locations in A
    # 2) A_test:  train on A, apply to test locations in A
    # 3) B_train: train on A, apply to same train locations in B
    # 4) B_test:  train on A, apply to test locations in B
    a_train_r2: list[float]
    a_test_r2: list[float]
    b_train_r2: list[float]
    b_test_r2: list[float]
    a_preds: list[np.ndarray]
    b_preds: list[np.ndarray]
    per_channel_a_train: list[np.ndarray] | None
    per_channel_a_test: list[np.ndarray] | None
    per_channel_b_train: list[np.ndarray] | None
    per_channel_b_test: list[np.ndarray] | None


def _eval_per_channel_r2(
    Z: np.ndarray,
    y_true: np.ndarray,
    w: np.ndarray,
    y_mean: np.ndarray,
) -> np.ndarray:
    """Vectorized evaluation of 1D per-channel linear probes.

    Z: (n_patches, c), y_true: (n_patches, d), w: (c, d), y_mean: (d,)
    Returns: per-channel R^2 array of shape (c,).
    """
    pred = Z[:, :, None] * w[None, :, :] + y_mean[None, None, :]  # (n_patches, c, d)
    ss_res = ((y_true[:, None, :] - pred) ** 2).sum(axis=0)  # (c, d)
    ss_tot = ((y_true - y_true.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)[None, :]  # (1, d)
    return (1.0 - ss_res / ss_tot).mean(axis=-1)


def do_cross_image_linear_probe(
    features: list[np.ndarray],
    ramp_type: RampTypes,
    mask_step: int = 6,
    mask_cutoff_frac: float = 1.0,
    random_mask: bool = True,
    probe_by_channel: bool = True,
    scale_feats: bool = True,
    regressor: Regressor = "ridge",
    all_to_all: bool = False,
) -> CrossImageProbeResult:
    """Train a linear probe once per image A and evaluate on A (train/test) and B (train/test).

    If `all_to_all=False`, pairs each image i with j = (i + 1) % N (quick option over N images).
    If `all_to_all=True`, evaluates each probe trained on image i across all j != i.
    """
    n_imgs = len(features)
    h, w, c = features[0].shape
    target = get_ramp(ramp_type, h, w)
    d = target.shape[-1]

    a_train_r2: list[float] = []
    a_test_r2: list[float] = []
    b_train_r2: list[float] = []
    b_test_r2: list[float] = []
    a_preds: list[np.ndarray] = []
    b_preds: list[np.ndarray] = []

    per_channel_a_train: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_a_test: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_b_train: list[np.ndarray] | None = [] if probe_by_channel else None
    per_channel_b_test: list[np.ndarray] | None = [] if probe_by_channel else None

    for i in range(n_imgs):
        feats_A = features[i]
        sample_mask = gen_sample_mask(
            (h, w), ramp_type, step=mask_step, cutoff_frac=mask_cutoff_frac, random_mask=random_mask
        )

        X_A_train = feats_A[sample_mask, :]
        X_A_test = feats_A[~sample_mask, :]
        X_A_full = feats_A.reshape(-1, c)
        y_train = target[sample_mask, :]
        y_test = target[~sample_mask, :]

        scaler = None
        if scale_feats:
            a_scaler = StandardScaler().fit(X_A_full)
            # scaler = StandardScaler().fit(X_A_train)
            X_A_train_sc = a_scaler.transform(X_A_train)
            X_A_test_sc = a_scaler.transform(X_A_test)
            X_A_full_sc = a_scaler.transform(X_A_full)
        else:
            X_A_train_sc = X_A_train
            X_A_test_sc = X_A_test
            X_A_full_sc = X_A_full

        # Train once on Image A
        reg = get_reg_model(regressor)
        reg.fit(X_A_train_sc, y_train)

        # 1) Apply to train locations in A & 2) Apply to test locations in A
        a_train_r2.append(float(reg.score(X_A_train_sc, y_train)))
        a_test_r2.append(float(reg.score(X_A_test_sc, y_test)))
        a_preds.append(reg.predict(X_A_full_sc).reshape(h, w, d))

        # Optional per-channel probe trained once on Image A
        if probe_by_channel:
            alpha = 1.0 if regressor == "ridge" else 0.0
            y_mean = y_train.mean(axis=0)
            denom = (X_A_train_sc ** 2).sum(axis=0) + alpha
            w_ch = np.einsum("nc,nd->cd", X_A_train_sc, y_train - y_mean[None, :]) / denom[:, None]
            assert per_channel_a_train is not None and per_channel_a_test is not None
            per_channel_a_train.append(_eval_per_channel_r2(X_A_train_sc, y_train, w_ch, y_mean))
            per_channel_a_test.append(_eval_per_channel_r2(X_A_test_sc, y_test, w_ch, y_mean))

        # Target image(s) B
        b_indices = [j for j in range(n_imgs) if j != i] if all_to_all else [(i + 1) % n_imgs]
        b_pred_accum = np.zeros((h, w, d), dtype=np.float64)

        for j in b_indices:
            feats_B = features[j]
            X_B_train = feats_B[sample_mask, :]
            X_B_test = feats_B[~sample_mask, :]
            X_B_full = feats_B.reshape(-1, c)

            if scale_feats:
                b_scaler = StandardScaler().fit(X_B_full)
                X_B_train_sc = b_scaler.transform(X_B_train)
                X_B_test_sc = b_scaler.transform(X_B_test)
                X_B_full_sc = b_scaler.transform(X_B_full)
            else:
                X_B_train_sc = X_B_train
                X_B_test_sc = X_B_test
                X_B_full_sc = X_B_full

            # 3) Apply to same train locations in B & 4) Apply to test locations in B
            b_train_r2.append(float(reg.score(X_B_train_sc, y_train)))
            b_test_r2.append(float(reg.score(X_B_test_sc, y_test)))
            b_pred_accum += reg.predict(X_B_full_sc).reshape(h, w, d) / len(b_indices)

            if probe_by_channel:
                assert per_channel_b_train is not None and per_channel_b_test is not None
                per_channel_b_train.append(_eval_per_channel_r2(X_B_train_sc, y_train, w_ch, y_mean))
                per_channel_b_test.append(_eval_per_channel_r2(X_B_test_sc, y_test, w_ch, y_mean))

        b_preds.append(b_pred_accum)

    return {
        "a_train_r2": a_train_r2,
        "a_test_r2": a_test_r2,
        "b_train_r2": b_train_r2,
        "b_test_r2": b_test_r2,
        "a_preds": a_preds,
        "b_preds": b_preds,
        "per_channel_a_train": per_channel_a_train,
        "per_channel_a_test": per_channel_a_test,
        "per_channel_b_train": per_channel_b_train,
        "per_channel_b_test": per_channel_b_test,
    }

In [6]:
ONLY_LR_UD: bool = False  # Set True to only run 'lr+ud'; False to run ('lr', 'ud', 'diag', 'radial') + 'lr+ud'
ALL_TO_ALL: bool = False  # Set False for quick i -> (i+1)%N over the N images; True for all-to-all pairs (i, j)
SCALE_FEATS: bool = True
MASK_CUTOFF_FRAC = 1.0
STEP = 6
RANDOM_MASK = True

fig_ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramps_to_run: tuple[RampTypes, ...] = ('lr+ud',) if ONLY_LR_UD else (*fig_ramps, 'lr+ud')
ramps_to_results: dict[RampTypes, CrossImageProbeResult] = {}

mode_str = f"all-to-all ({n_imgs * (n_imgs - 1)} pairs)" if ALL_TO_ALL else f"i -> j ({n_imgs} pairs)"
print(f"Running cross-image linear probe [{mode_str}] on {n_imgs} images:")
print(f"{'Ramp':<8} | {'1) A_train':<15} | {'2) A_test':<15} | {'3) B_train':<15} | {'4) B_test':<15}")
print("-" * 78)

for ramp in ramps_to_run:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    res = do_cross_image_linear_probe(
        features,
        ramp,
        mask_step=STEP,
        mask_cutoff_frac=MASK_CUTOFF_FRAC,
        random_mask=RANDOM_MASK,
        probe_by_channel=(ramp in fig_ramps and not ONLY_LR_UD),
        scale_feats=SCALE_FEATS,
        regressor="ridge",
        all_to_all=ALL_TO_ALL,
    )
    ramps_to_results[ramp] = res

    a_tr_m, a_tr_s = np.mean(res['a_train_r2']), np.std(res['a_train_r2'])
    a_te_m, a_te_s = np.mean(res['a_test_r2']), np.std(res['a_test_r2'])
    b_tr_m, b_tr_s = np.mean(res['b_train_r2']), np.std(res['b_train_r2'])
    b_te_m, b_te_s = np.mean(res['b_test_r2']), np.std(res['b_test_r2'])
    print(
        f"{ramp:<8} | "
        f"{a_tr_m:6.2f} ± {a_tr_s:5.2f} | "
        f"{a_te_m:6.2f} ± {a_te_s:5.2f} | "
        f"{b_tr_m:6.2f} ± {b_tr_s:5.2f} | "
        f"{b_te_m:6.2f} ± {b_te_s:5.2f}"
    )

Running cross-image linear probe [i -> j (60 pairs)] on 60 images:
Ramp     | 1) A_train      | 2) A_test       | 3) B_train      | 4) B_test      
------------------------------------------------------------------------------
lr       |   1.00 ±  0.00 |   0.88 ±  0.06 |   0.78 ±  0.10 |   0.76 ±  0.09
ud       |   1.00 ±  0.00 |   0.93 ±  0.03 |   0.83 ±  0.09 |   0.81 ±  0.08
diag     |   1.00 ±  0.00 |   0.91 ±  0.04 |   0.81 ±  0.08 |   0.78 ±  0.07
radial   |   1.00 ±  0.00 |   0.90 ±  0.04 |   0.63 ±  0.17 |   0.63 ±  0.16
lr+ud    |   1.00 ±  0.00 |   0.90 ±  0.04 |   0.81 ±  0.08 |   0.79 ±  0.07


In [7]:
def average_cross_results(
    res: CrossImageProbeResult,
    eval_set: str = "b_test",
) -> tuple[np.ndarray | None, np.ndarray | None, float, float, np.ndarray]:
    """Aggregate per-channel scores, stack R^2, and mean prediction on image B."""
    r2_key = f"{eval_set}_r2"
    ch_key = f"per_channel_{eval_set}"

    scores_arr = np.array(res[r2_key])  # type: ignore[index]
    mean_score = float(np.mean(scores_arr))
    std_score = float(np.std(scores_arr))

    preds_list = res["a_preds"] if eval_set.startswith("a_") else res["b_preds"]
    n_pred_dims = preds_list[0].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))
    for pred_arr in preds_list:
        pred = resize(pred_arr, mean_pred.shape, order=1)
        mean_pred += pred / len(preds_list)

    ch_scores = res[ch_key]  # type: ignore[index]
    if ch_scores is None:
        return None, None, mean_score, std_score, mean_pred

    channel_scores_arr = np.array(ch_scores)
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)
    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred


def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none', lw=0.25)
    ax.add_collection(pc)

In [8]:
if not ONLY_LR_UD:
    n_rows, n_cols = 4, 9
    W, H = 7.5, 1 * 2.3
    fig = plt.figure(figsize=(W, H))

    plt.style.use("thesis.mplstyle")
    plt.rcParams['text.usetex'] = False
    add_custom_font('resources/fonts', 'Grotesk')

    w_spacing = [1 for _ in range(n_cols)]
    FIG_B_COL_OFFSET = 1
    FIG_B_W_COLS = 2
    FIG_C_COL_OFFSET = FIG_B_COL_OFFSET + FIG_B_W_COLS + 2
    w_spacing[4] = 0.05
    w_spacing[FIG_C_COL_OFFSET:] = [0.7] * len(w_spacing[FIG_C_COL_OFFSET:])

    gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, wspace=0.2, hspace=0.35)

    dino_colors: dict[RampTypes, str] = {
        'lr': '#5762D5',
        'ud': '#6370C0',
        'diag': '#6E7DAB',
        'radial': '#575366',
    }
    alibi_colors: dict[RampTypes, str] = {
        'lr': '#16ce37',
        'ud': '#87f1ab',
        'diag': '#5f978b',
        'radial': '#133b32',
    }
    vit_b_colors: dict[RampTypes, str] = {
        'lr': '#FFE048',
        'ud': '#CBCA74',
        'diag': '#969765',
        'radial': '#2D3047',
    }
    model_to_ramp_colours: dict[str, dict[RampTypes, str]] = {
        "dinov2_s": dino_colors,
        "alibi_dv2_coco": alibi_colors,
        "vit_b": vit_b_colors,
    }
    colors = model_to_ramp_colours.get(selected_model, dino_colors)

    ramp_to_title: dict[RampTypes, str] = {
        'lr': 'Left-right',
        'ud': 'Up-down',
        'diag': 'Diagonal',
        'radial': 'Radial',
    }

    top_left_ramp_ax = None
    for row, ramp in enumerate(fig_ramps):
        h, w = 34, 34
        ramp_arr = get_ramp(ramp, h, w)
        ramp_ax = fig.add_subplot(gs[row, 0])
        ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

        mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        add_red_square_overlay(ramp_ax, mask, 1, 1)

        ramp_ax.set_xticks([])
        ramp_ax.set_yticks([])
        ramp_ax.set_ylabel(ramp_to_title[ramp], fontsize=7)

        if row == 0:
            ramp_ax.set_title('Target ramp')
            top_left_ramp_ax = ramp_ax

    worst_channels = []
    for row, ramp in enumerate(fig_ramps):
        ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET:FIG_B_COL_OFFSET + FIG_B_W_COLS])
        mean_channel_scores, std_channel_scores, mean_b_test, _, mean_pred = average_cross_results(
            ramps_to_results[ramp], eval_set="b_test"
        )
        mean_b_train = float(np.mean(ramps_to_results[ramp]["b_train_r2"]))
        assert mean_channel_scores is not None and std_channel_scores is not None

        print(f"{ramp} (B_test top channels): {np.argsort(-mean_channel_scores)[:6]}")
        worst_channels.extend(list(np.argsort(-mean_channel_scores)[:2]))

        ax.hlines(0, 0, n_dims, 'red', '--', lw=0.5)
        ax.plot(mean_channel_scores, color=colors[ramp], lw=0.5)
        ax.fill_between(
            np.arange(n_dims),
            mean_channel_scores - std_channel_scores,
            mean_channel_scores + std_channel_scores,
            color=colors[ramp],
            alpha=0.3,
        )
        ax.set_ylim(-0.1, 1)
        ax.set_xlim(0 - 10, n_dims + 10)
        ax.tick_params(axis='both', labelsize=6)
        if row != len(fig_ramps) - 1:
            ax.set_xticklabels([])
        ax.grid(alpha=0.2)

        mean_pred_ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET + FIG_B_W_COLS])
        mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)

        if row == 0:
            ax.set_title('Per-channel ' + r'$R^2$' + ' scores')
            mean_pred_ax.set_title('Mean prediction (B)')
        elif row == len(fig_ramps) - 1:
            ax.set_xlabel('Channel')
            mean_pred_ax.set_xlabel('')

        mean_pred_ax.set_ylabel(
            rf'$R^2_{{\mathrm{{tr}}}}:${mean_b_train:.2f}' + '\n' + rf'$R^2_{{\mathrm{{te}}}}:${mean_b_test:.2f}',
            fontsize=6,
        )
        mean_pred_ax.set_xticks([])
        mean_pred_ax.set_yticks([])

    bad_channel_map = {
        'dv2': [55, 89, 228],
        'dvt': [55, 113, 188],
        'dv2_b': [39, 354, 480],
        'vit_b': [18, 390, 599],
        'dv': [293, 61, 89],
        'dv_b': [432, 266, 99],
        'dv3': [149, 123, 74],
        'clip_b': [69, 526, 554],
        'eva02': [180, 192, 180],
        'sam_b': [50, 27, 210],
        'deit': [173, 99, 302],
        'alibi_dv2_cb': [78, 73, 228],
        'alibi_dv2_coco': [118, 228, 359],
    }
    bad_channels = bad_channel_map.get(selected_model, [47, 117, 359])

    top_right_ax = None
    for i, img in enumerate(preview_imgs):
        img_ax = fig.add_subplot(gs[i, FIG_C_COL_OFFSET])
        img_ax.imshow(img)
        img_ax.set_xticks([])
        img_ax.set_yticks([])
        if i == 0:
            top_right_ax = img_ax

        for j, ch in enumerate(bad_channels):
            ax = fig.add_subplot(gs[i, FIG_C_COL_OFFSET + j + 1])
            feats = preview_feats[i]
            selected_ch = feats[:, :, ch]
            ax.imshow(selected_ch, cmap='viridis')
            if i == 0:
                ax.set_title(f'Ch. {ch}')
            ax.set_xticks([])
            ax.set_yticks([])

    labels = ['(a)', '(b)']
    axes = [top_left_ramp_ax, top_right_ax]
    for i in range(2):
        ax = axes[i]
        assert ax is not None
        x = -1.1 if i == 0 else -0.3
        ax.text(x, 1.3, labels[i], transform=ax.transAxes, fontweight='bold', color='black')

    SAVE = True
    if SAVE:
        plt.savefig("saved/R01.pdf", dpi=300, bbox_inches='tight')
        plt.close()

findfont: Failed to find font weight normal for Host Grotesk, now using 300.


lr (B_test top channels): [117 359 183  78 168 169]
ud (B_test top channels): [114  30 359 261 328 380]
diag (B_test top channels): [117 114  78 380 183 332]
radial (B_test top channels): [ 47  55  89 113  26  56]


findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
